In [1]:
#!/usr/bin/env python
# coding: utf-8

# =============================================================================
# 导入库，加载数据和CV设置
# =============================================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle
import json
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import callbacks
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.random.set_seed(42)

# 创建输出文件夹
Path('models/dnn').mkdir(parents=True, exist_ok=True)

# 加载数据
df = pd.read_csv('data/full_dataset.csv')

with open('data/cv_setup.pkl', 'rb') as f:
    cv_setup = pickle.load(f)
with open('data/cv_setup_inner.pkl', 'rb') as f:
    cv_setup_inner = pickle.load(f)

feature_cols = cv_setup['feature_cols']
target_col   = cv_setup['target_col']
outer_folds  = cv_setup['outer_folds']

X = df[feature_cols].values
y = df[target_col].values

# 超参数搜索组合
param_combinations = [
    {'hidden_layers': [64, 32],     'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 32},
    {'hidden_layers': [64, 32],     'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 64},
    {'hidden_layers': [64, 32],     'dropout_rate': 0.2, 'learning_rate': 0.01,  'batch_size': 32},
    {'hidden_layers': [64, 32, 16], 'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 32},
    {'hidden_layers': [32, 16],     'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 32},
    {'hidden_layers': [64, 32],     'dropout_rate': 0.3, 'learning_rate': 0.001, 'batch_size': 32},
]

# 训练配置
EPOCHS   = 200
PATIENCE = 20
VERBOSE  = 0

# 边缘原子和原子类型配置
EDGE_THRESHOLD   = 0.1
ATOM_TYPE_COLS   = ['T_O', 'T_C', 'T_Ti1', 'T_Ti2']
ATOM_TYPE_LABELS = ['O', 'C', 'Ti_inner', 'Ti_outer']


# =============================================================================
# 辅助函数
# =============================================================================

def create_dnn_model(input_dim, hidden_layers, dropout_rate, learning_rate):
    model = Sequential()
    model.add(Dense(hidden_layers[0], activation='relu', input_dim=input_dim))
    model.add(Dropout(dropout_rate))
    for units in hidden_layers[1:]:
        model.add(Dense(units, activation='relu'))
        model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation='linear'))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse', metrics=['mae'])
    return model

def calculate_metrics(y_true, y_pred):
    return {
        'mae':       float(mean_absolute_error(y_true, y_pred)),
        'rmse':      float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'r2':        float(r2_score(y_true, y_pred)),
        'n_samples': int(len(y_true))
    }

def classify_and_evaluate(y_true, y_pred, df_subset):
    results = {}
    results['overall'] = calculate_metrics(y_true, y_pred)

    edge_mask     = df_subset['D_E'].values < EDGE_THRESHOLD
    interior_mask = ~edge_mask

    results['edge']     = calculate_metrics(y_true[edge_mask],     y_pred[edge_mask])     if edge_mask.sum()     > 0 else {'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'n_samples': 0}
    results['interior'] = calculate_metrics(y_true[interior_mask], y_pred[interior_mask]) if interior_mask.sum() > 0 else {'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'n_samples': 0}

    results['by_atom_type'] = {}
    for col, label in zip(ATOM_TYPE_COLS, ATOM_TYPE_LABELS):
        mask = df_subset[col].values == 1
        results['by_atom_type'][label] = calculate_metrics(y_true[mask], y_pred[mask]) if mask.sum() > 0 \
            else {'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'n_samples': 0}

    return results




In [2]:
# =============================================================================
# 嵌套交叉验证主循环
# =============================================================================

outer_results         = []
best_models           = []
all_train_predictions = []
all_test_predictions  = []
training_histories    = []
classification_results = {'fold_wise': {}, 'averaged': {}}

start_time = time.time()

for fold_idx, outer_fold in enumerate(outer_folds):
    train_idx = outer_fold['train_atoms_idx']
    test_idx  = outer_fold['test_atoms_idx']

    X_train = X[train_idx];  y_train = y[train_idx]
    X_test  = X[test_idx];   y_test  = y[test_idx]

    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_test  = df.iloc[test_idx].reset_index(drop=True)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # 内层CV超参数搜索
    inner_folds = cv_setup_inner[fold_idx]['inner_folds']
    best_params      = None
    best_inner_score = float('inf')

    for params in param_combinations:
        inner_scores = []
        for inner_fold in inner_folds:
            inner_train_set = set(inner_fold['inner_train_structures'])
            inner_val_set   = set(inner_fold['inner_val_structures'])
            struct_ids = df.iloc[train_idx]['structure_id'].values
            inner_train_mask = np.array([s in inner_train_set for s in struct_ids])
            inner_val_mask   = np.array([s in inner_val_set   for s in struct_ids])

            X_i_train = X_train_scaled[inner_train_mask]
            y_i_train = y_train[inner_train_mask]
            X_i_val   = X_train_scaled[inner_val_mask]
            y_i_val   = y_train[inner_val_mask]

            model = create_dnn_model(X.shape[1], params['hidden_layers'],
                                     params['dropout_rate'], params['learning_rate'])
            model.fit(X_i_train, y_i_train,
                      validation_data=(X_i_val, y_i_val),
                      epochs=EPOCHS, batch_size=params['batch_size'],
                      callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE,
                                                         restore_best_weights=True, verbose=0)],
                      verbose=VERBOSE)
            inner_scores.append(model.evaluate(X_i_val, y_i_val, verbose=0)[0])

        mean_score = np.mean(inner_scores)
        if mean_score < best_inner_score:
            best_inner_score = mean_score
            best_params = params

    # 用最优参数在外层训练集上训练
    best_model = create_dnn_model(X.shape[1], best_params['hidden_layers'],
                                  best_params['dropout_rate'], best_params['learning_rate'])
    history = best_model.fit(
        X_train_scaled, y_train,
        epochs=EPOCHS, batch_size=best_params['batch_size'],
        callbacks=[callbacks.EarlyStopping(monitor='loss', patience=PATIENCE,
                                           restore_best_weights=True, verbose=0)],
        verbose=VERBOSE
    )
    training_histories.append({'fold': fold_idx + 1, 'history': history.history})

    y_train_pred = best_model.predict(X_train_scaled, verbose=0).flatten()
    y_test_pred  = best_model.predict(X_test_scaled,  verbose=0).flatten()

    train_class_results = classify_and_evaluate(y_train, y_train_pred, df_train)
    test_class_results  = classify_and_evaluate(y_test,  y_test_pred,  df_test)

    print(f"折叠 {fold_idx+1} | 最优参数: {best_params} | "
          f"训练MAE: {train_class_results['overall']['mae']:.5f} | "
          f"测试MAE: {test_class_results['overall']['mae']:.5f} | "
          f"epochs: {len(history.history['loss'])}")

    fold_result = {
        'fold': fold_idx + 1,
        'best_params': best_params,
        'train_metrics': train_class_results,
        'test_metrics':  test_class_results,
        'n_train': len(train_idx),
        'n_test':  len(test_idx),
        'n_epochs': len(history.history['loss']),
        'train_structures': outer_fold['train_structures'],
        'test_structures':  outer_fold['test_structures']
    }
    outer_results.append(fold_result)
    best_models.append({'fold': fold_idx + 1, 'model': best_model,
                        'scaler': scaler, 'best_params': best_params})

    classification_results['fold_wise'][f'fold_{fold_idx+1}'] = {
        'train': train_class_results,
        'test':  test_class_results
    }

    for i in range(len(train_idx)):
        all_train_predictions.append({
            'fold': fold_idx + 1,
            'atom_index': int(train_idx[i]),
            'structure_id': df_train.iloc[i]['structure_id'],
            'D_E': float(df_train.iloc[i]['D_E']),
            'T_O': int(df_train.iloc[i]['T_O']),
            'T_C': int(df_train.iloc[i]['T_C']),
            'T_Ti1': int(df_train.iloc[i]['T_Ti1']),
            'T_Ti2': int(df_train.iloc[i]['T_Ti2']),
            'y_true': float(y_train[i]),
            'y_pred': float(y_train_pred[i]),
            'error': float(y_train_pred[i] - y_train[i]),
            'abs_error': float(abs(y_train_pred[i] - y_train[i]))
        })
    for i in range(len(test_idx)):
        all_test_predictions.append({
            'fold': fold_idx + 1,
            'atom_index': int(test_idx[i]),
            'structure_id': df_test.iloc[i]['structure_id'],
            'D_E': float(df_test.iloc[i]['D_E']),
            'T_O': int(df_test.iloc[i]['T_O']),
            'T_C': int(df_test.iloc[i]['T_C']),
            'T_Ti1': int(df_test.iloc[i]['T_Ti1']),
            'T_Ti2': int(df_test.iloc[i]['T_Ti2']),
            'y_true': float(y_test[i]),
            'y_pred': float(y_test_pred[i]),
            'error': float(y_test_pred[i] - y_test[i]),
            'abs_error': float(abs(y_test_pred[i] - y_test[i]))
        })

elapsed_time = time.time() - start_time
print(f"\n嵌套CV完成，耗时 {elapsed_time:.1f} 秒 ({elapsed_time/60:.1f} 分钟)")




折叠 1 | 最优参数: {'hidden_layers': [64, 32], 'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 64} | 训练MAE: 0.01439 | 测试MAE: 0.01517 | epochs: 93
折叠 2 | 最优参数: {'hidden_layers': [64, 32], 'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 64} | 训练MAE: 0.01827 | 测试MAE: 0.01863 | epochs: 100
折叠 3 | 最优参数: {'hidden_layers': [64, 32], 'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 64} | 训练MAE: 0.01883 | 测试MAE: 0.01905 | epochs: 108
折叠 4 | 最优参数: {'hidden_layers': [64, 32], 'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 64} | 训练MAE: 0.02967 | 测试MAE: 0.02949 | epochs: 96
折叠 5 | 最优参数: {'hidden_layers': [64, 32], 'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 32} | 训练MAE: 0.02707 | 测试MAE: 0.02810 | epochs: 85

嵌套CV完成，耗时 8068.2 秒 (134.5 分钟)


In [3]:
# =============================================================================
# 计算平均性能，训练最终模型，保存所有结果
# =============================================================================

def calculate_averaged_metrics(outer_results, dataset_type='test'):
    key = f'{dataset_type}_metrics'
    averaged = {}

    for category in ['overall', 'edge', 'interior']:
        maes  = [r[key][category]['mae']  for r in outer_results]
        rmses = [r[key][category]['rmse'] for r in outer_results]
        r2s   = [r[key][category]['r2']   for r in outer_results]
        ns    = [r[key][category]['n_samples'] for r in outer_results]
        averaged[category] = {
            'mae_mean':  float(np.mean(maes)),  'mae_std':  float(np.std(maes)),
            'rmse_mean': float(np.mean(rmses)), 'rmse_std': float(np.std(rmses)),
            'r2_mean':   float(np.mean(r2s)),   'r2_std':   float(np.std(r2s)),
            'n_samples_mean': float(np.mean(ns))
        }

    averaged['by_atom_type'] = {}
    for label in ATOM_TYPE_LABELS:
        maes  = [r[key]['by_atom_type'][label]['mae']  for r in outer_results]
        rmses = [r[key]['by_atom_type'][label]['rmse'] for r in outer_results]
        r2s   = [r[key]['by_atom_type'][label]['r2']   for r in outer_results]
        ns    = [r[key]['by_atom_type'][label]['n_samples'] for r in outer_results]
        averaged['by_atom_type'][label] = {
            'mae_mean':  float(np.mean(maes)),  'mae_std':  float(np.std(maes)),
            'rmse_mean': float(np.mean(rmses)), 'rmse_std': float(np.std(rmses)),
            'r2_mean':   float(np.mean(r2s)),   'r2_std':   float(np.std(r2s)),
            'n_samples_mean': float(np.mean(ns))
        }

    return averaged

train_averaged = calculate_averaged_metrics(outer_results, 'train')
test_averaged  = calculate_averaged_metrics(outer_results, 'test')
classification_results['averaged'] = {'train': train_averaged, 'test': test_averaged}

# 打印平均性能
print("\n5折CV平均性能（测试集）:")
for cat in ['overall', 'edge', 'interior']:
    m = test_averaged[cat]
    print(f"  {cat:10s} - MAE: {m['mae_mean']:.5f} ± {m['mae_std']:.5f}  "
          f"RMSE: {m['rmse_mean']:.5f} ± {m['rmse_std']:.5f}  "
          f"R²: {m['r2_mean']:.6f} ± {m['r2_std']:.6f}")

# 最终模型（使用测试集MAE最小的折叠的超参数）
best_fold_idx = int(np.argmin([r['test_metrics']['overall']['mae'] for r in outer_results]))
final_params  = outer_results[best_fold_idx]['best_params']
print(f"\n最终模型参数（来自折叠 {best_fold_idx+1}）: {final_params}")

scaler_final   = StandardScaler()
X_scaled_final = scaler_final.fit_transform(X)

final_model = create_dnn_model(X.shape[1], final_params['hidden_layers'],
                               final_params['dropout_rate'], final_params['learning_rate'])
final_history = final_model.fit(
    X_scaled_final, y,
    epochs=EPOCHS, batch_size=final_params['batch_size'],
    callbacks=[callbacks.EarlyStopping(monitor='loss', patience=PATIENCE,
                                       restore_best_weights=True, verbose=0)],
    verbose=VERBOSE
)
print(f"最终模型训练完成，epochs: {len(final_history.history['loss'])}")

# 保存预测CSV
df_train_pred_all = pd.DataFrame(all_train_predictions)
df_test_pred_all  = pd.DataFrame(all_test_predictions)

for fold_num in range(1, 6):
    df_train_pred_all[df_train_pred_all['fold'] == fold_num].to_csv(
        f'models/dnn/fold_{fold_num}_train_predictions.csv', index=False)
    df_test_pred_all[df_test_pred_all['fold'] == fold_num].to_csv(
        f'models/dnn/fold_{fold_num}_test_predictions.csv', index=False)

# 保存分类评分CSV
classification_rows = []
for fold_num in range(1, 6):
    fold_data = classification_results['fold_wise'][f'fold_{fold_num}']
    for dataset in ['train', 'test']:
        for category in ['overall', 'edge', 'interior']:
            d = fold_data[dataset][category]
            classification_rows.append({
                'fold': fold_num, 'dataset': dataset,
                'category': category, 'subcategory': 'all',
                'mae': d['mae'], 'rmse': d['rmse'],
                'r2': d['r2'], 'n_samples': d['n_samples']
            })
        for label in ATOM_TYPE_LABELS:
            d = fold_data[dataset]['by_atom_type'][label]
            classification_rows.append({
                'fold': fold_num, 'dataset': dataset,
                'category': 'atom_type', 'subcategory': label,
                'mae': d['mae'], 'rmse': d['rmse'],
                'r2': d['r2'], 'n_samples': d['n_samples']
            })

pd.DataFrame(classification_rows).to_csv(
    'models/dnn/classification_scores.csv', index=False)

# 保存模型文件（DNN用Keras格式保存）
final_model.save('models/dnn/dnn_model.keras')

model_package = {
    'scaler': scaler_final,
    'feature_cols': feature_cols,
    'target_col': target_col,
    'best_params': final_params,
    'model_path': 'models/dnn/dnn_model.keras',
    'cv_performance': {'train': train_averaged, 'test': test_averaged},
    'edge_threshold': EDGE_THRESHOLD,
    'atom_type_labels': ATOM_TYPE_LABELS,
    'training_config': {'epochs': EPOCHS, 'patience': PATIENCE}
}
with open('models/dnn/dnn_model_package.pkl', 'wb') as f:
    pickle.dump(model_package, f)

# 保存每折模型
all_folds_info = []
for model_info in best_models:
    fold_num = model_info['fold']
    model_info['model'].save(f'models/dnn/fold_{fold_num}_model.keras')
    all_folds_info.append({
        'fold': fold_num,
        'model_path': f'models/dnn/fold_{fold_num}_model.keras',
        'scaler': model_info['scaler'],
        'best_params': model_info['best_params']
    })

with open('models/dnn/dnn_all_folds.pkl', 'wb') as f:
    pickle.dump(all_folds_info, f)

results_dict = {
    'model_name': 'Deep Neural Network',
    'n_folds': len(outer_folds),
    'n_features': len(feature_cols),
    'training_time_seconds': float(elapsed_time),
    'edge_threshold': EDGE_THRESHOLD,
    'architecture': {
        'param_combinations': param_combinations,
        'max_epochs': EPOCHS,
        'early_stopping_patience': PATIENCE
    },
    'cv_results': {'train': train_averaged, 'test': test_averaged},
    'fold_details': outer_results,
    'training_histories': training_histories,
    'final_model': {
        'params': final_params,
        'n_epochs': len(final_history.history['loss'])
    }
}
with open('models/dnn/dnn_results.json', 'w') as f:
    json.dump(results_dict, f, indent=4)

print("\n已保存: 预测CSV, classification_scores.csv, dnn_model.keras, "
      "dnn_model_package.pkl, fold_*_model.keras, dnn_all_folds.pkl, dnn_results.json")


5折CV平均性能（测试集）:
  overall    - MAE: 0.02209 ± 0.00566  RMSE: 0.03047 ± 0.00474  R²: 0.998541 ± 0.000448
  edge       - MAE: 0.03966 ± 0.00285  RMSE: 0.05844 ± 0.00771  R²: 0.992831 ± 0.001838
  interior   - MAE: 0.01998 ± 0.00636  RMSE: 0.02480 ± 0.00580  R²: 0.999034 ± 0.000431

最终模型参数（来自折叠 1）: {'hidden_layers': [64, 32], 'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 64}
最终模型训练完成，epochs: 105

已保存: 预测CSV, classification_scores.csv, dnn_model.keras, dnn_model_package.pkl, fold_*_model.keras, dnn_all_folds.pkl, dnn_results.json
